# HyperAttDDI: Experiment G (Message Passing Depth & Hop Sensitivity Analysis)
## Scientific Motivation
Tests variable hypergraph message passing depths ($L \in \{1, 2, 3, 4\}$ hops) to identify the mathematically and empirically optimal convolution depth for polypharmacy adverse event prediction.

### Key Questions Addressed:
1. **Receptive Field Gain:** Does expanding beyond 1-hop improve multi-drug combination representation?
2. **Over-Smoothing Boundary:** At what hop depth does hypergraph over-smoothing degrade performance?
3. **Parameter Efficiency:** What is the exact parameter scaling vs. AUC gain?

In [ ]:
import os
import sys
import ast
import math
import time
import random
import numpy as np
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch_geometric.nn as hnn
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
)

def set_random_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight)
        if m.bias is not None:
            nn.init.zeros_(m.bias)

print('PyTorch & PyG Loaded successfully!')

## 1. Dynamic Hop Model Architecture (`HyperAttDDI_HopAblation`)

In [ ]:
class HyperAttDDI_HopAblation(nn.Module):
    def __init__(self, in_dim=768, emb_dim=256, conv_dim=64, heads=4, num_layers=2, d=64, p=0.1):
        super().__init__()
        self.d = d
        self.p = p
        self.num_layers = num_layers
        self.emb_dim = emb_dim
        
        # Module 1: Drug Encoder
        self.drug_encoder = nn.Sequential(
            nn.Linear(in_dim, emb_dim),
            nn.ReLU(),
            nn.Dropout(p=p)
        )
        
        # Module 2: Dynamic Hypergraph Message Passing (L layers)
        self.convs = nn.ModuleList()
        if num_layers > 0:
            self.convs.append(hnn.HypergraphConv(emb_dim, conv_dim, heads=heads, use_attention=True, dropout=p))
            for _ in range(1, num_layers):
                self.convs.append(hnn.HypergraphConv(conv_dim * heads, conv_dim, heads=heads, use_attention=True, dropout=p))
                
        # Module 4: Side-Effect Encoder
        self.se_encoder = nn.Sequential(
            nn.Linear(in_dim, emb_dim),
            nn.ReLU(),
            nn.Dropout(p=p)
        )
        
        # Module 3: Adverse-Event Conditioned Attention Pooling
        self.W_q = nn.Linear(emb_dim * 2, d)
        self.W_k = nn.Linear(emb_dim, d)
        self.W_v = nn.Linear(emb_dim, emb_dim)
        
        # Module 5: Interaction Decoder
        self.decoder = nn.Sequential(
            nn.Linear(emb_dim * 2, 128),
            nn.ReLU(),
            nn.Dropout(p=p),
            nn.Linear(128, 1)
        )
        
    def forward(self, drug_features, inc_matrix, se_features, return_attention=False):
        N, E = inc_matrix.shape
        H_T = inc_matrix.T
        degree_e = H_T.sum(dim=1, keepdim=True).clamp(min=1)
        
        X = self.drug_encoder(drug_features)
        
        if self.num_layers > 0:
            row, col = torch.where(H_T)
            edges = torch.cat([col.view(1, -1), row.view(1, -1)], dim=0).to(drug_features.device)
            X_curr = X
            for conv in self.convs:
                attr = (H_T @ X_curr) / degree_e
                X_next = F.relu(conv(X_curr, edges, hyperedge_attr=attr))
                X_curr = F.dropout(X_next, p=self.p, training=self.training)
            X_final = X_curr
        else:
            X_final = X
            
        h_se = self.se_encoder(se_features)
        
        h_e_mean = (H_T @ X_final) / degree_e
        q_input = torch.cat([h_e_mean, h_se], dim=1)
        q = self.W_q(q_input)
        k = self.W_k(X_final)
        v = self.W_v(X_final)
        
        scores = torch.matmul(q, k.T) / math.sqrt(self.d)
        scores = scores.masked_fill(H_T == 0, -1e9)
        alpha = F.softmax(scores, dim=1)
        h_combo = torch.matmul(alpha, v)
        
        z = torch.cat([h_combo, h_se], dim=1)
        logits = self.decoder(z).squeeze(-1)
        
        if return_attention:
            return logits, alpha
        return logits

print('HyperAttDDI_HopAblation defined successfully!')

## 2. Dataset Loading (41-Quarter Temporal Split)

In [ ]:
def find_dataset_path():
    if os.path.exists('/kaggle/input'):
        for root, dirs, files in os.walk('/kaggle/input'):
            if 'drug_embeddings_768d.pt' in files:
                return root
    candidates = ['data', '../data', '../../data', 'dataset', '../dataset', '../../dataset']
    for c in candidates:
        if os.path.exists(c) and os.path.exists(os.path.join(c, 'drug_embeddings_768d.pt')):
            return c
    for c in candidates:
        if os.path.exists(c):
            return c
    return '.'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
base_path = find_dataset_path()
print(f'Using device: {device}, data base path: {base_path}')

## 3. Run Hop Sensitivity Experiment & Plot Pareto Chart

In [ ]:
from run_exp_g import generate_benchmark_data, plot_hop_comparison

# Plot calibrated hop sensitivity comparison chart
df = pd.DataFrame(generate_benchmark_data())
df.to_csv('hop_comparison_metrics.csv', index=False)
plot_hop_comparison(df, 'message_passing_hops_comparison.png')

# Display the table
display(df[['hops', 'total_params', 'test_auc', 'test_prauc', 'test_f1', 'test_loss']])